# LexiCore — GPU training on Colab

Fine-tunes the three DistilBERT encoders (speaker, multi-label claim, stance) on the full 2782-segment dataset with 5-fold CV, then prints the Macro-F1 numbers to compare against the TF-IDF baselines.

**How to run:** Runtime → Change runtime type → **GPU (A100)** → then Runtime → **Run all**. Takes ~20–40 min. No API key needed — the data is already in the repo.

Baselines to beat (TF-IDF, full data):
- Speaker Macro-F1 **0.661**
- Claim multi-label Micro-F1 **0.493** / Macro **0.365**
- Stance Macro-F1 **0.572**

In [ ]:
# 1) Confirm we have a GPU (expect A100 / T4)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2) Get the code + data (already committed in the repo)
%cd /content
![ -d lexicore-agi-discourse ] && rm -rf lexicore-agi-discourse
!git clone --depth 1 https://github.com/yunuseozcelik/lexicore-agi-discourse.git
%cd lexicore-agi-discourse

In [ ]:
# 3) Train all three encoders. A100 -> BATCH=64. (T4 -> use BATCH=16.)
#    Runs speaker, claim, and stance DistilBERT (5-fold each); logs to gpu_results/.
!BATCH=64 MAXLEN=256 bash run_gpu.sh

In [ ]:
# 4) Pull out just the headline numbers to copy back
import glob
for f in sorted(glob.glob('gpu_results/*.txt')):
    print('=' * 60)
    print(f)
    print('=' * 60)
    for line in open(f, encoding='utf-8'):
        if any(k in line for k in ('Macro-F1', 'Micro-F1', 'macro avg', 'micro avg', 'accuracy', '===')):
            print(line.rstrip())

## After it finishes

Copy the printed Macro-F1 / Micro-F1 lines and send them back — we'll drop them into `PROGRESS.md` as the DistilBERT-vs-baseline comparison (the last open experiment for the paper).

*(Optional — save the logs to your Drive so they persist:)*
```python
from google.colab import drive; drive.mount('/content/drive')
!cp -r gpu_results /content/drive/MyDrive/lexicore_gpu_results
```